In [1]:
import os
import pandas as pd
import requests
import spotipy
from IPython.display import display
from spotipy.oauth2 import SpotifyClientCredentials, SpotifyOAuth
from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
loaded = load_dotenv(dotenv_path=dotenv_path, override=True)
print(f"dotenv_loaded={loaded}, dotenv_path={dotenv_path}")

dotenv_loaded=True, dotenv_path=/Users/Marcy_Student/Desktop/Spotify_Project/.env


In [2]:
# setting up credentials
def read_env(name: str, required: bool = True) -> str:
    raw = os.getenv(name)
    value = (raw or "").strip().strip('"').strip("'")
    if required and not value:
        raise ValueError(f"Missing required environment variable: {name}")
    return value

spotify_client_id = read_env("SPOTIFY_CLIENT_ID")
spotify_client_secret = read_env("SPOTIFY_CLIENT_SECRET")
rapidapi_key = read_env("RAPIDAPI_KEY", required=False)
rapidapi_host = "track-analysis.p.rapidapi.com"

auth_manager = SpotifyClientCredentials(
    client_id=spotify_client_id,
    client_secret=spotify_client_secret,
    requests_timeout=15,
    cache_handler=None,
    requests_session=True,
    proxies=None,
)
sp = spotipy.Spotify(auth_manager=auth_manager)

headers = {
    "x-rapidapi-key": rapidapi_key,
    "x-rapidapi-host": rapidapi_host,
    "Content-Type": "application/json",
}

print("Spotify + RapidAPI configuration loaded")
print(f"SPOTIFY_CLIENT_ID length={len(spotify_client_id)}")
print(f"SPOTIFY_CLIENT_SECRET length={len(spotify_client_secret)}")

Spotify + RapidAPI configuration loaded
SPOTIFY_CLIENT_ID length=32
SPOTIFY_CLIENT_SECRET length=32


In [3]:
# testing spotipy with my fav artist NF
from IPython.display import display
query = "artist:NF"
album_results = sp.search(q=query, type="album", limit=10)
track_results = sp.search(q=query, type="track", limit=10)
albums = [
    {
        "album": item["name"],
        "album_type": item["album_type"],
        "release_date": item["release_date"],
        "total_tracks": item["total_tracks"],
    }
    for item in album_results["albums"]["items"]
]
tracks = [
    {

        "track": item["name"],
        "album": item["album"]["name"],
        "artists": ", ".join(artist["name"] for artist in item["artists"]),
        "preview_url": item.get("preview_url"),
    }
    for item in track_results["tracks"]["items"]
]
print("NF albums")
display(pd.DataFrame(albums))
print("NF tracks")
display(pd.DataFrame(tracks))


SpotifyOauthError: error: invalid_client, error_description: Invalid client

In [ ]:
# creating a function to get the track informations of tracks_id
# My track Id will be store in a list
# This is a simple test for one track id
# the workflow will take export batch of track ids from spotify playlists endpoint\
track_ids = [
    "7s25THrKz86DM225dOYwnr"  # sample ID from SoundNet docs
]

def get_track_info(track_id):
    track = sp.track(track_id)

    return{
        "track_id": track_id,
        "track_name": track["name"],
        "artist": ",".join(a["name"] for a in track["artists"]),
        "album":track["album"]["name"],
    }
tracks_df = pd.DataFrame([get_track_info(track_id) for track_id in track_ids])
display(tracks_df)

In [ ]:
# Get track IDs and metadata from Spotify top playlists.
# Playlist item endpoints require user OAuth in this app context.

playlist_ids = [
    "37i9dQZEVXbLRQDuF5jeBp",  # Top 50 - USA
    "37i9dQZEVXbMDoHDwVN2tF",  # Top 50 - Global
    "37i9dQZEVXbKuaTI1Z1Afx",  # Viral 50 - Global
]
tracks_per_playlist = 50

oauth_cache_path = ".spotify_oauth_cache"
top_playlist_tracks_df = pd.DataFrame(columns=["track_id", "track_name", "artist", "album"])

try:
    redirect_uri = read_env("SPOTIFY_REDIRECT_URI")

    user_auth_manager = SpotifyOAuth(
        client_id=spotify_client_id,
        client_secret=spotify_client_secret,
        redirect_uri=redirect_uri,
        scope="playlist-read-private playlist-read-collaborative",
        open_browser=True,
        cache_path=oauth_cache_path,
        show_dialog=True,
    )
    user_sp = spotipy.Spotify(auth_manager=user_auth_manager)

    rows = []
    for playlist_id in playlist_ids:
        playlist_tracks = user_sp.playlist_items(
            playlist_id,
            limit=tracks_per_playlist,
            additional_types=("track",),
        )

        for item in playlist_tracks.get("items", []):
            track = item.get("track") if item else None
            if not track or not track.get("id"):
                continue

            rows.append(
                {
                    "track_id": track.get("id"),
                    "track_name": track.get("name"),
                    "artist": ", ".join(a["name"] for a in track.get("artists", [])),
                    "album": track.get("album", {}).get("name"),
                }
            )

    top_playlist_tracks_df = pd.DataFrame(rows).drop_duplicates(subset=["track_id"]).reset_index(drop=True)
except Exception as exc:
    print(f"Could not fetch playlist tracks with current auth: {exc}")
    print("OAuth checklist:")
    print("1) Add SPOTIFY_REDIRECT_URI to .env (example: http://127.0.0.1:8888/callback)")
    print("2) Add the same exact Redirect URI in Spotify Developer Dashboard > app settings")
    print("3) Ensure client ID and secret are from that same Spotify app")
    print("4) Delete .spotify_oauth_cache if you recently switched apps/secrets")

track_ids = top_playlist_tracks_df["track_id"].tolist() if "track_id" in top_playlist_tracks_df.columns else []
tracks_df = top_playlist_tracks_df.copy()

display(top_playlist_tracks_df)
print(f"Collected {len(track_ids)} unique track IDs from chart playlists.")

ValueError: Missing required environment variable: SPOTIFY_REDIRECT_URI

In [ ]:
merged_df = tracks_df.merge(analysis_df, on="track_id", how="left")

merged_df.head()